# 04 — Feature Extraction & Selection
> Engineers new features from spatial, ecoregion, and dC/dN decomposition data.
> Evaluates and ranks features using variance filters, correlation, mutual information,
> and Random Forest importances.

**Prerequisite:** `03_preprocessing.ipynb`

## 0 · Colab Setup

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone https://github.com/YOUR_USERNAME/tree_carbon_ml.git 2>/dev/null || true
    %cd tree_carbon_ml
    !pip install -r requirements.txt -q
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('Ready')

## 1 · Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

from features.engineer import FeatureEngineer
from features.selector import FeatureSelector
from visualization.plots import plot_feature_importance

pd.set_option('display.float_format', '{:.4f}'.format)
TARGET = 'TPH.gs.dC.dN0.01'
TARGET_LOG = 'target_log'

df = pd.read_parquet('data/interim/03_preprocessed.parquet')
with open('data/interim/03_column_manifest.json') as f:
    manifest = json.load(f)

print(f'Loaded: {df.shape}')
print('Manifest keys:', list(manifest.keys()))

## 2 · Engineer New Features
Run the `FeatureEngineer` from `src/features/engineer.py` to add all derived features.

In [ ]:
engineer = FeatureEngineer(df)
df_feat = engineer.build_all()

new_features = engineer.get_feature_cols()
print(f'New features created ({len(new_features)}):')
for f in new_features:
    print(f'  {f}')

### 2a · Inspect Engineered Features

In [ ]:
display(df_feat[new_features].describe().round(4))

In [ ]:
# Distribution of key engineered features
plot_cols = [c for c in [
    'feat_gs_ratio', 'feat_expanded_per_ha', 'feat_log_magnitude',
    'feat_lat_bin', 'feat_lon_bin', 'feat_eco3_target_mean'
] if c in df_feat.columns]

n = len(plot_cols)
fig, axes = plt.subplots(2, (n+1)//2, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    data = df_feat[col].replace([np.inf, -np.inf], np.nan).dropna()
    data = data.clip(data.quantile(0.01), data.quantile(0.99))
    axes[i].hist(data, bins=50, color='#5C6BC0', edgecolor='white', lw=0.3)
    axes[i].set_title(col, fontsize=9)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Engineered Feature Distributions', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/09_engineered_features.png', dpi=150, bbox_inches='tight')
plt.show()

### 2b · Additional Manual Features

In [ ]:
# Interaction: latitude × ecoregion L1 encoded
if 'LAT' in df_feat.columns and 'feat_enc_NA_L1CODE' in df_feat.columns:
    df_feat['feat_lat_x_eco1'] = df_feat['LAT'] * df_feat['feat_enc_NA_L1CODE']
    print('Added feat_lat_x_eco1')

# Ratio of growth to net signal
if 'TPH.g.dC.dN0.01' in df_feat.columns:
    denom = df_feat[TARGET].replace(0, np.nan)
    df_feat['feat_growth_share'] = df_feat['TPH.g.dC.dN0.01'] / denom
    df_feat['feat_growth_share'] = df_feat['feat_growth_share'].clip(-10, 10)
    print('Added feat_growth_share')

# Distance from equator (absolute latitude)
if 'LAT' in df_feat.columns:
    df_feat['feat_abs_lat'] = df_feat['LAT'].abs()
    print('Added feat_abs_lat')

# Ecoregion area proxy
if 'eco.EXPN.ha' in df_feat.columns and 'state.EXPN.ha' in df_feat.columns:
    df_feat['feat_eco_state_ratio'] = (
        df_feat['eco.EXPN.ha'] / df_feat['state.EXPN.ha'].replace(0, np.nan)
    ).clip(0, 100)
    print('Added feat_eco_state_ratio')

# Update feature list
all_feat_cols = [c for c in df_feat.columns if c.startswith('feat_')]
print(f'\nTotal engineered features: {len(all_feat_cols)}')

## 3 · Feature Selection Pipeline

In [ ]:
# Build X matrix — use only feat_ columns + core numeric (cleaned)
feature_candidates = all_feat_cols + manifest.get('scaled_cols', [])
feature_candidates = [c for c in feature_candidates if c in df_feat.columns]

# Replace inf and fill NaN
X = df_feat[feature_candidates].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df_feat[TARGET_LOG].fillna(0)

print(f'Feature matrix X: {X.shape}')
print(f'Target vector  y: {y.shape}')

### 3a · Variance Filter

In [ ]:
selector = FeatureSelector(X, y)
kept_variance = selector.variance_filter(threshold=0.001)
print(f'Features after variance filter: {len(kept_variance)}')

### 3b · Correlation Filter

In [ ]:
X_var = X[kept_variance]
sel2 = FeatureSelector(X_var, y)
kept_corr = sel2.correlation_filter(threshold=0.95)
print(f'Features after correlation filter: {len(kept_corr)}')

### 3c · SelectKBest — Mutual Information

In [ ]:
X_corr = X_var[kept_corr]
sel3 = FeatureSelector(X_corr, y)
top_mi = sel3.select_k_best(k=min(15, len(kept_corr)), method='mutual_info_regression')

### 3d · Random Forest Feature Importance

In [ ]:
print('Fitting Random Forest for feature importance (may take 1–2 min)...')
importances = sel3.random_forest_importance(n_estimators=150, top_n=20)

fig = plot_feature_importance(importances, top_n=20, save=True)
plt.show()

### 3e · SHAP Values (optional — requires shap package)

In [ ]:
try:
    import shap
    from sklearn.ensemble import RandomForestRegressor

    print('Computing SHAP values on top-20 features...')
    top20 = importances.head(20).index.tolist()
    X_top = X_corr[top20].fillna(0)

    rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    rf.fit(X_top, y)

    explainer = shap.TreeExplainer(rf)
    sample_idx = X_top.sample(min(500, len(X_top)), random_state=42).index
    shap_values = explainer.shap_values(X_top.loc[sample_idx])

    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_top.loc[sample_idx], plot_type='bar', show=False)
    plt.title('SHAP Feature Importance (mean |SHAP|)')
    plt.tight_layout()
    plt.savefig('outputs/figures/10_shap_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_top.loc[sample_idx], show=False)
    plt.title('SHAP Beeswarm Plot')
    plt.tight_layout()
    plt.savefig('outputs/figures/10b_shap_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.show()

except ImportError:
    print('shap not installed — run: pip install shap')

## 4 · Final Feature Set Selection

In [ ]:
# Take union of top-MI and top-RF features, plus baseline numeric
top_rf_20 = importances.head(20).index.tolist()
top_mi_15 = top_mi

# Always include core geographic features if present
baseline = [c for c in [
    'LAT_scaled','LON_scaled','EXPN.ha_scaled',
    'eco.EXPN.ha_scaled','state.EXPN.ha_scaled'
] if c in X.columns]

final_features = list(set(top_rf_20 + top_mi_15 + baseline))
final_features = [f for f in final_features if f in df_feat.columns]

print(f'Final feature set: {len(final_features)} features')
print(sorted(final_features))

In [ ]:
# Correlation heatmap of final features
X_final = df_feat[final_features].replace([np.inf,-np.inf], np.nan).fillna(0)
corr = X_final.corr()

fig, ax = plt.subplots(figsize=(max(10, len(final_features)*0.6), max(8, len(final_features)*0.5)))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0, linewidths=0.3, ax=ax)
ax.set_title('Final Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('outputs/figures/11_final_feature_corr.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 · Save Feature-Enriched Dataset

In [ ]:
os.makedirs('data/interim', exist_ok=True)

out_path = 'data/interim/04_features.parquet'
df_feat.to_parquet(out_path, index=False)
print(f'Full enriched dataset saved: {out_path}  shape={df_feat.shape}')

# Save the final feature list
feature_manifest = {
    'final_features': sorted(final_features),
    'target': TARGET,
    'target_log': TARGET_LOG,
    'target_binary': 'target_binary',
    'target_class3': 'target_class3',
    'id_col': 'PLT_CN',
    'n_features': len(final_features),
    'n_rows': len(df_feat),
}
with open('data/interim/04_feature_manifest.json','w') as f:
    json.dump(feature_manifest, f, indent=2)
print('Feature manifest saved to data/interim/04_feature_manifest.json')
print(json.dumps(feature_manifest, indent=2))

---
### ✅ Notebook 04 Complete
Next: **05_model_ready_dataset.ipynb**